# Insturctions to follow before executing this script

## Step 1: Create the Isolated phase2_trellis (Run in Terminal)



### Tunnel for Jupyter
ssh -f -N -L 5402:scn33-mn:8888 rahkus@login.npsf.cdac.in
### Tunnel for Gradio
ssh -f -N -L 7862:scn33-mn:7860 rahkus@login.npsf.cdac.in


In [1]:
# 1. THE FROZEN RULE: Core Config First
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 2. ISOLATE GPU: Lock this entire notebook to a specific GPU to prevent VRAM collisions
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Change to 0, 1, 2, or 3
print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/files
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/info
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.
[SYSTEM] Notebook strictly locked to GPU: 0


In [2]:
!kill 9 2386856

/bin/bash: line 1: kill: (9) - No such process
/bin/bash: line 1: kill: (2386856) - No such process


In [3]:
!nvidia-smi

Sat Aug 22 23:21:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:07:00.0 Off |                    0 |
| N/A   29C    P0             55W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import os
from storage_manager import storage_preflight_path

ok, reason = storage_preflight_path(
    os.path.expanduser("~/aireos_cache"),
    required_gb=1.0,
    require_fresh_quota=True,
    minimum_free_gb=0.2,
)
if not ok:
    raise RuntimeError(f"Jupyter utility upgrade denied by storage policy: {reason}")

!pip install --upgrade jupyter ipywidgets


# Step 1: TRELLIS.2 (Setup)
Full C++ Compilation & DependenciesThis cell contains the complete, native compilation logic. If it detects you already built the extensions, it will safely skip to save 15 minutes. If you ever need to rebuild, change **FORCE_RECOMPILE = True**.

In [ ]:
import os
import sys
import subprocess
import shutil
import importlib.util
import hashlib
from core_config import SYSTEM_DIRS
from storage_manager import storage_preflight_path

FORCE_RECOMPILE = False

# All dependency installs/builds are performed on the compute node, but they
# still consume the shared user quota. Admit the setup phase before pip/conda.
setup_budget_gb = max(
    1.0,
    float(os.environ.get("AIEROS_TRELLIS_SETUP_BUDGET_GB", "8.0")),
)
ok, reason = storage_preflight_path(
    SYSTEM_DIRS["cache"],
    required_gb=setup_budget_gb,
    require_fresh_quota=True,
    minimum_free_gb=0.5,
)
if not ok:
    raise RuntimeError(f"TRELLIS setup denied by storage policy: {reason}")

def run_cmd(cmd, cwd=None):
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, cwd=cwd, check=True)

def is_installed(pkg_name):
    return importlib.util.find_spec(pkg_name) is not None

print("1. Verifying PyTorch 2.6.0 (CUDA 12.4)...")
run_cmd([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("1b. Installing CUDA-enabled xFormers matched to PyTorch 2.6.0+cu124...")
run_cmd([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", "--no-cache-dir", "xformers==0.0.29.post2", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("Verifying CUDA-enabled xFormers operators...")
xformers_info = subprocess.run(
    [sys.executable, "-m", "xformers.info"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)
print(xformers_info.stdout)
if xformers_info.returncode != 0:
    raise RuntimeError("xformers.info failed; attention backend validation could not run.")
xformers_fields = {}
for _line in xformers_info.stdout.splitlines():
    if ":" in _line:
        _key, _value = _line.split(":", 1)
        xformers_fields[_key.strip()] = _value.strip()
if (
    xformers_fields.get("build.cuda_version") != "1204"
    or xformers_fields.get("pytorch.cuda") != "available"
    or xformers_fields.get("memory_efficient_attention.cutlassF-pt") != "available"
):
    raise RuntimeError(
        "xFormers lacks the CUDA 12.4 attention kernels required by TRELLIS.2. "
        "Use xformers==0.0.29.post2 from https://download.pytorch.org/whl/cu124."
    )
print("XFORMERS_CUDA_VALIDATION=PASS")


print("2. Verifying ML and 3D dependencies...")
run_cmd([
    sys.executable, "-m", "pip", "install", 
    "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless", 
    "ninja", "trimesh", "transformers==4.56.1", "gradio", "tensorboard", "pandas", 
    "lpips", "zstandard", "kornia", "timm", "einops", "safetensors", "fire", 
    "accelerate", "ipywidgets"
])
run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"])

def prepare_extension_source(repo_dir, repo_url, revision, *, exact_commit=None, tag_name=None):
    """Create/reuse a clean versioned checkout without deleting existing source."""
    existed_before = os.path.exists(repo_dir)
    if existed_before:
        if not os.path.isdir(os.path.join(repo_dir, ".git")):
            raise RuntimeError(
                f"Extension source path exists but is not a Git checkout: {repo_dir}. "
                "Refusing to overwrite or delete it; inspect it manually."
            )
        remote = subprocess.check_output(
            ["git", "-C", repo_dir, "remote", "get-url", "origin"], text=True
        ).strip().lower()
        normalized_remote = remote.replace("git@github.com:", "https://github.com/").removesuffix(".git")
        expected_remote = repo_url.lower().removesuffix(".git")
        if normalized_remote != expected_remote:
            raise RuntimeError(
                f"Unexpected origin at {repo_dir}: {remote!r}; expected {repo_url!r}. "
                "Refusing to modify existing source."
            )
        dirty = subprocess.check_output(
            ["git", "-C", repo_dir, "status", "--porcelain"], text=True
        ).strip()
        if dirty:
            raise RuntimeError(
                f"Existing extension checkout is dirty: {repo_dir}. "
                "Refusing to discard changes; inspect it manually."
            )
    else:
        run_cmd(["git", "clone", "--recursive", repo_url, repo_dir])

    if exact_commit:
        current = subprocess.check_output(
            ["git", "-C", repo_dir, "rev-parse", "HEAD"], text=True
        ).strip()
        if current != exact_commit:
            if existed_before:
                raise RuntimeError(
                    f"Existing versioned checkout {repo_dir} is at {current}, "
                    f"expected {exact_commit}; refusing to change it."
                )
            run_cmd(["git", "-C", repo_dir, "checkout", "--detach", exact_commit])
            run_cmd(["git", "-C", repo_dir, "submodule", "update", "--init", "--recursive"])
        verified = subprocess.check_output(
            ["git", "-C", repo_dir, "rev-parse", "HEAD"], text=True
        ).strip()
        if verified != exact_commit:
            raise RuntimeError(f"Revision verification failed for {repo_dir}: {verified}")
    elif tag_name:
        tag_check = subprocess.run(
            ["git", "-C", repo_dir, "describe", "--tags", "--exact-match", "HEAD"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if tag_check.returncode != 0 or tag_check.stdout.strip() != tag_name:
            if existed_before:
                raise RuntimeError(
                    f"Existing versioned checkout {repo_dir} is not pinned to tag {tag_name}; "
                    "refusing to change it."
                )
            run_cmd(["git", "-C", repo_dir, "checkout", "--detach", tag_name])
            run_cmd(["git", "-C", repo_dir, "submodule", "update", "--init", "--recursive"])
        verified_tag = subprocess.check_output(
            ["git", "-C", repo_dir, "describe", "--tags", "--exact-match", "HEAD"], text=True
        ).strip()
        if verified_tag != tag_name:
            raise RuntimeError(f"Tag verification failed for {repo_dir}: {verified_tag}")
    return repo_dir

extensions_exist = (
    is_installed("nvdiffrast")
    and is_installed("nvdiffrec")
    and is_installed("flex_gemm")
    and is_installed("o_voxel")
)

if not FORCE_RECOMPILE and extensions_exist:
    print("3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.")
else:
    print("3. Compiling versioned TRELLIS.2 C++ extension sources without deleting existing checkouts...")
    extension_budget_gb = max(1.0, float(os.environ.get("AIEROS_TRELLIS_EXTENSION_BUDGET_GB", "8.0")))
    ok, reason = storage_preflight_path(
        SYSTEM_DIRS["cache"],
        required_gb=extension_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=0.5,
    )
    if not ok:
        raise RuntimeError(f"TRELLIS extension build denied by storage policy: {reason}")

    EXT_DIR = os.path.join(SYSTEM_DIRS["cache"], "trellis2_extensions")
    os.makedirs(EXT_DIR, exist_ok=True)

    nd_dir = os.path.join(EXT_DIR, "nvdiffrast-v0.4.0")
    prepare_extension_source(
        nd_dir, "https://github.com/NVlabs/nvdiffrast.git", "v0.4.0", tag_name="v0.4.0"
    )
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", nd_dir, "--no-build-isolation"])

    nr_dir = os.path.join(EXT_DIR, "nvdiffrec-renderutils-b296927")
    prepare_extension_source(
        nr_dir, "https://github.com/JeffreyXiang/nvdiffrec.git",
        "b296927cc7fd01c2ac1087c8065c4d7248f72da4", exact_commit="b296927cc7fd01c2ac1087c8065c4d7248f72da4"
    )
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", nr_dir, "--no-build-isolation"])

    fg_dir = os.path.join(EXT_DIR, "FlexGEMM-6dd94a8")
    prepare_extension_source(
        fg_dir, "https://github.com/JeffreyXiang/FlexGEMM.git",
        "6dd94a859c26ee8246888502eada3dd8ad85532e", exact_commit="6dd94a859c26ee8246888502eada3dd8ad85532e"
    )
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", fg_dir, "--no-build-isolation"])

# ==============================================================================
# 3b. PIN A KNOWN-GOOD OFFICIAL CuMesh REVISION
# ==============================================================================
# Importability alone is insufficient: stale CuMesh builds can crash during
# UV atlas/GLB export. Install once per Python environment, gated by storage
# admission. Keep old source trees intact; use a versioned sibling checkout.
CUMESH_PINNED_REVISION = "12289e1062f0603f2f0d0771b02e1395d247f26f"
EXT_DIR = os.path.join(SYSTEM_DIRS["cache"], "trellis2_extensions")
os.makedirs(EXT_DIR, exist_ok=True)
CUMESH_SOURCE_DIR = os.path.join(EXT_DIR, "CuMesh-12289e1062f0")
_env_id = hashlib.sha256(sys.prefix.encode("utf-8")).hexdigest()[:12]
CUMESH_INSTALL_MARKER = os.path.join(
    EXT_DIR, f"cumesh-{_env_id}-12289e1062f0.installed"
)

def _verify_cumesh_source(path):
    remote = subprocess.check_output(
        ["git", "-C", path, "remote", "get-url", "origin"], text=True
    ).strip().lower().replace("git@github.com:", "https://github.com/").removesuffix(".git")
    if remote != "https://github.com/JeffreyXiang/CuMesh":
        raise RuntimeError(f"Unexpected CuMesh origin at {path}: {remote}; refusing to build.")
    revision = subprocess.check_output(
        ["git", "-C", path, "rev-parse", "HEAD"], text=True
    ).strip()
    dirty = subprocess.check_output(
        ["git", "-C", path, "status", "--porcelain"], text=True
    ).strip()
    if revision != CUMESH_PINNED_REVISION or dirty:
        raise RuntimeError(
            f"Refusing to overwrite CuMesh source at {path}: "
            f"revision={revision}, dirty={bool(dirty)}. "
            "Inspect it manually and choose a new versioned directory."
        )

if FORCE_RECOMPILE or not os.path.isfile(CUMESH_INSTALL_MARKER):
    _ok, _reason = storage_preflight_path(
        SYSTEM_DIRS["cache"],
        required_gb=max(1.0, float(os.environ.get("AIEROS_TRELLIS_CUMESH_BUILD_GB", "8.0"))),
        require_fresh_quota=True,
        minimum_free_gb=0.5,
    )
    if not _ok:
        raise RuntimeError(f"CuMesh build denied by storage policy: {_reason}")

    if not os.path.exists(CUMESH_SOURCE_DIR):
        run_cmd([
            "git", "clone", "--recursive",
            "https://github.com/JeffreyXiang/CuMesh.git",
            CUMESH_SOURCE_DIR,
        ])
        run_cmd(["git", "-C", CUMESH_SOURCE_DIR, "fetch", "origin", CUMESH_PINNED_REVISION])
        run_cmd(["git", "-C", CUMESH_SOURCE_DIR, "checkout", "--detach", CUMESH_PINNED_REVISION])
        run_cmd(["git", "-C", CUMESH_SOURCE_DIR, "submodule", "update", "--init", "--recursive"])
    elif not os.path.isdir(os.path.join(CUMESH_SOURCE_DIR, ".git")):
        raise RuntimeError(
            f"CuMesh source path exists but is not a git checkout: {CUMESH_SOURCE_DIR}; "
            "refusing to delete or replace it."
        )
    else:
        _verify_cumesh_source(CUMESH_SOURCE_DIR)

    print(f"Building official CuMesh revision {CUMESH_PINNED_REVISION}...")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "--force-reinstall", "--no-deps", "--no-cache-dir", "--no-build-isolation",
        CUMESH_SOURCE_DIR,
    ])
    # Marker is written only after a successful build/install.
    with open(CUMESH_INSTALL_MARKER, "w", encoding="utf-8") as _marker_file:
        _marker_file.write(CUMESH_PINNED_REVISION + "\n")
    import cumesh as _cumesh_check
    print(
        "CUMESH_PINNED_BUILD=PASS",
        f"revision={CUMESH_PINNED_REVISION}",
        f"module={getattr(_cumesh_check, '__file__', '?')}",
    )
else:
    with open(CUMESH_INSTALL_MARKER, "r", encoding="utf-8") as _marker_file:
        _installed_cumesh_revision = _marker_file.read().strip()
    if _installed_cumesh_revision != CUMESH_PINNED_REVISION:
        raise RuntimeError(
            f"CuMesh install marker mismatch: {_installed_cumesh_revision!r}; "
            "refusing to skip validation."
        )
    print(f"CuMesh revision already built for this environment: {CUMESH_PINNED_REVISION}")

t2_dir = os.path.join(SYSTEM_DIRS["models"], "TRELLIS.2")
if not os.path.exists(t2_dir):
    # Clone only when the final path is absent. Existing/partial source is never
    # removed or overwritten; model/cache admission was checked above.
    run_cmd([
        "git", "clone", "-b", "main",
        "https://github.com/microsoft/TRELLIS.2.git", t2_dir, "--recursive"
    ])
elif not os.path.isdir(os.path.join(t2_dir, ".git")):
    raise RuntimeError(
        f"TRELLIS.2 path exists but is not a Git checkout: {t2_dir}. "
        "Refusing to delete or replace it; inspect it manually."
    )
else:
    _trellis_remote = subprocess.check_output(
        ["git", "-C", t2_dir, "remote", "get-url", "origin"], text=True
    ).strip().lower().removesuffix(".git")
    if "microsoft/trellis.2" not in _trellis_remote:
        raise RuntimeError(
            f"Unexpected TRELLIS.2 checkout origin: {_trellis_remote}. "
            "Refusing to change existing source."
        )
    _trellis_dirty = subprocess.check_output(
        ["git", "-C", t2_dir, "status", "--porcelain"], text=True
    ).strip()
    if _trellis_dirty:
        raise RuntimeError(
            f"TRELLIS.2 source checkout has local changes: {t2_dir}. "
            "Refusing to discard them; inspect manually."
        )
    if not os.path.isfile(os.path.join(t2_dir, "setup.sh")):
        raise RuntimeError(
            f"TRELLIS.2 checkout is incomplete at {t2_dir}; refusing to overwrite it."
        )
    print(f"Reusing verified official TRELLIS.2 checkout at {t2_dir}")


if FORCE_RECOMPILE or not is_installed("o_voxel"):
    print("   Installing o-voxel...")
    run_cmd([sys.executable, "-m", "pip", "install", os.path.join(t2_dir, "o-voxel"), "--no-build-isolation"])

sys.path.insert(0, t2_dir)
print("Environment Ready! TRELLIS.2 mapped and ready.")

# Step 2: FLUX.2 (Setup)
Install and validate the FLUX.2 Diffusers dependencies before loading either pipeline.


In [ ]:
import sys
import subprocess

print("Installing official Diffusers 0.37.0 + compatible Hub/Transformers pins for FLUX.2 Klein...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-U",
    "diffusers==0.37.0",
    "huggingface-hub==0.36.0",
    "transformers==4.56.1",
], check=True)

# Fail at setup time if package constraints are unresolved; don't leave this
# until a user clicks Generate and receives a long lazy-import traceback.
subprocess.run([
    sys.executable, "-c",
    (
        "import importlib.metadata as m; "
        "from packaging.version import Version; "
        "hub=Version(m.version('huggingface-hub')); "
        "assert Version('0.34.0') <= hub < Version('1.0.0'), hub; "
        "from diffusers import DiffusionPipeline, Flux2KleinPipeline; "
        "print('FLUX_IMPORT_CHECK=PASS', 'huggingface_hub='+str(hub), "
        "'diffusers='+m.version('diffusers'))"
    ),
], check=True)
print("FLUX.2 dependencies installed and import-checked.")

from packaging.version import Version
_loaded_hub = sys.modules.get("huggingface_hub")
if _loaded_hub is not None:
    _loaded_hub_version = Version(getattr(_loaded_hub, "__version__", "0"))
    if not (Version("0.34.0") <= _loaded_hub_version < Version("1.0.0")):
        raise RuntimeError(
            f"FLUX dependencies were updated on disk, but this Jupyter kernel still "
            f"has huggingface-hub {_loaded_hub_version} loaded in memory. "
            "Restart the kernel, then execute the notebook again from the top. "
            "Do not continue to TRELLIS/FLUX inference in this stale kernel."
        )


# Step 3: TRELLIS.2 (Interactive Studio - Image Upload)


In [ ]:
import core_config
from core_config import SYSTEM_DIRS
from huggingface_hub import snapshot_download
try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

import os
import sys
import torch
import random
import gc
from PIL import Image
import gradio as gr

from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

os.environ["ATTN_BACKEND"] = "xformers"
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["TRELLIS_CACHE_DIR"] = os.path.join(SYSTEM_DIRS['models'], "trellis_weights")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "3d_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_trellis_pipeline():
    model_id = "microsoft/TRELLIS.2-4B"
    # Hugging Face uses <models>/hub as the cache directory under core_config.
    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")
    try:
        local_model_dir = snapshot_download(
            repo_id=model_id,
            cache_dir=hub_cache_dir,
            local_files_only=True,
        )
        print(f"TRELLIS cache hit; resolved local snapshot: {local_model_dir}")
    except LocalEntryNotFoundError as cache_error:
        # Only a genuine local-cache miss enters download admission. API, import,
        # model-load and CUDA errors are deliberately not treated as cache misses.
        required_gb = (
            estimate_hf_repo_bytes(model_id) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        print(
            f"TRELLIS cache miss ({type(cache_error).__name__}). "
            f"Storage admission budget: {required_gb:.2f} GB"
        )
        with managed_storage_operation(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),
        ):
            local_model_dir = snapshot_download(
                repo_id=model_id,
                cache_dir=hub_cache_dir,
            )

    # Official TRELLIS.2 from_pretrained accepts a local path; local_files_only
    # belongs to snapshot_download, not Trellis2ImageTo3DPipeline.from_pretrained.
    return Trellis2ImageTo3DPipeline.from_pretrained(local_model_dir)
trellis_pipe = load_trellis_pipeline()
trellis_pipe.cuda()


@torch.inference_mode()
def extrude_3d_asset(input_image, decimation_target: int):
    if input_image is None:
        raise gr.Error("Please upload an image first!")
        
    seed = random.randint(0, 999999)
    print(f"\n[EXTRUDING] Processing uploaded image | Seed: {seed}")
    
    input_image = input_image.convert("RGB")
    mesh = trellis_pipe.run(input_image)[0]
    mesh.simplify(16777216) 
    
    glb_path = os.path.join(OUTPUT_DIR, f"standalone_t2_{seed}.glb")
    glb = o_voxel.postprocess.to_glb(
        vertices            =   mesh.vertices,
        faces               =   mesh.faces,
        attr_volume         =   mesh.attrs,
        coords              =   mesh.coords,
        attr_layout         =   mesh.layout,
        voxel_size          =   mesh.voxel_size,
        aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target   =   int(decimation_target),
        texture_size        =   4096,
        remesh              =   True,
        remesh_band         =   1,
        remesh_project      =   0,
        verbose             =   False
    )
    with managed_storage_operation(
        "outputs",
        required_gb=0.20,
        require_fresh_quota=False,
        minimum_free_gb=0.05,
    ):
        glb.export(glb_path, extension_webp=True)
    
    del mesh, glb
    gc.collect()
    torch.cuda.empty_cache()
    
    return glb_path

with gr.Blocks(title="TRELLIS.2 Standalone", theme=gr.themes.Monochrome()) as demo_trellis:
    gr.Markdown("# 🕹️ TRELLIS.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Upload 2D Concept", type="pil")
            decimation = gr.Slider(label="Target Polygons", minimum=100000, maximum=1000000, value=500000, step=10000)
            generate_btn = gr.Button("Extrude to 3D", variant="primary", size="lg")
        with gr.Column(scale=2):
            output_3d = gr.Model3D(label="3D Interactive Viewer")

    generate_btn.click(fn=extrude_3d_asset, inputs=[input_image, decimation], outputs=[output_3d])

print("Starting TRELLIS Server on port 7862...")
demo_trellis.launch(server_name="0.0.0.0", server_port=7862, inline=True, share=False)

# Step 4: FLUX.2 (Interactive Studio - Text to Image)


In [ ]:
import core_config
from core_config import SYSTEM_DIRS
from huggingface_hub import snapshot_download
try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

import os
import torch
import random
import gc
import gradio as gr
from diffusers import DiffusionPipeline

from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "images")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_flux_pipeline():
    model_id = "black-forest-labs/FLUX.2-klein-4B"
    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")
    kwargs = dict(
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        text_encoder_2=None,
        tokenizer_2=None,
        feature_extractor=None,
        image_encoder=None,
    )

    try:
        local_model_dir = snapshot_download(
            repo_id=model_id,
            cache_dir=hub_cache_dir,
            local_files_only=True,
        )
        print(f"FLUX cache hit; resolved local snapshot: {local_model_dir}")
    except LocalEntryNotFoundError as cache_error:
        # Only a true local cache miss enters storage admission. Don't mask import,
        # compatibility, model-config, or CUDA errors as missing model files.
        required_gb = (
            estimate_hf_repo_bytes(model_id) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        print(
            f"FLUX cache miss ({type(cache_error).__name__}). "
            f"Storage admission budget: {required_gb:.2f} GB"
        )
        with managed_storage_operation(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),
        ):
            local_model_dir = snapshot_download(
                repo_id=model_id,
                cache_dir=hub_cache_dir,
            )

    pipe = DiffusionPipeline.from_pretrained(local_model_dir, **kwargs)
    pipe.enable_model_cpu_offload()
    return pipe
flux_pipe = load_flux_pipeline()


@torch.inference_mode()
def generate_concept(prompt: str, seed: int, randomize: bool):
    if randomize: seed = random.randint(0, 999999)
    print(f"\n[GENERATING] Prompt: '{prompt}' | Seed: {seed}")
    
    generator = torch.Generator(device="cpu").manual_seed(seed)
    image = flux_pipe(
        prompt,
        guidance_scale=3.5, 
        num_inference_steps=28, 
        max_sequence_length=512,
        generator=generator
    ).images[0].convert("RGB")
    
    out_path = os.path.join(OUTPUT_DIR, f"flux2_concept_{seed}.png")
    with managed_storage_operation(
        "outputs",
        required_gb=0.05,
        require_fresh_quota=False,
        minimum_free_gb=0.02,
    ):
        image.save(out_path)
    
    gc.collect()
    torch.cuda.empty_cache()
    return image, seed

with gr.Blocks(title="FLUX.2 Standalone", theme=gr.themes.Monochrome()) as demo_flux:
    gr.Markdown("# 🎨 FLUX.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", value="A sci-fi crate", lines=4)
            seed = gr.Number(label="Seed", value=42, precision=0)
            randomize = gr.Checkbox(label="Randomize", value=True)
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column(scale=2):
            output_image = gr.Image(label="FLUX.2 Output")

    generate_btn.click(fn=generate_concept, inputs=[prompt, seed, randomize], outputs=[output_image, seed])

print("Starting FLUX Server on port 7863...")
demo_flux.launch(server_name="0.0.0.0", server_port=7863, inline=True, share=False)

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi